In [2]:
import json

ambiguous_jsons = ["annotations.json", "annotations_fixed.json"]

for json_file in ambiguous_jsons:
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)

        file_names = [img["file_name"] for img in data.get("images", [])]

        # Check for our clue files
        i_files = [name for name in file_names if name.startswith('i')]
        g_files = [name for name in file_names if name.startswith('g')]

        print(f"🔍 Inspecting: {json_file}")
        print(f"  -> Total images inside: {len(file_names)}")
        print(f"  -> Contains 'g' images: {len(g_files) > 0} (Found {len(g_files)})")
        print(f"  -> Contains 'i' images: {len(i_files) > 0} (Found {len(i_files)})\n")

    except FileNotFoundError:
        print(f"⚠️ Could not find {json_file}. Make sure it's in the same folder as this notebook.")

🔍 Inspecting: annotations.json
  -> Total images inside: 196
  -> Contains 'g' images: True (Found 196)
  -> Contains 'i' images: False (Found 0)

🔍 Inspecting: annotations_fixed.json
  -> Total images inside: 255
  -> Contains 'g' images: True (Found 196)
  -> Contains 'i' images: True (Found 59)



In [1]:
import json
import os
import shutil

# 1. Define your inputs (Make sure to update the folder paths!)
datasets = [
    {"json_path": "images400.json", "img_dir": "images400/new_images"},
    {"json_path": "instances_default.json", "img_dir": "val_anu"},
    {"json_path": "annotations.json", "img_dir": "train"},
    {"json_path": "annotations_fixed.json", "img_dir": "train2/train"}
]

# --- THE FIX: Create a dedicated master folder for everything ---
master_folder = "my_merged_coco_dataset"
os.makedirs(master_folder, exist_ok=True)

# Set paths to be inside the master folder
output_json = os.path.join(master_folder, "merged_dataset.json")
output_img_dir = os.path.join(master_folder, "images")

# Create the images sub-directory
os.makedirs(output_img_dir, exist_ok=True)

merged_data = {
    "info": {"description": "Merged COCO Dataset"},
    "images": [],
    "annotations": [],
    "categories": []
}

global_img_id = 1
global_ann_id = 1

for ds_idx, ds in enumerate(datasets):
    print(f"Processing {ds['json_path']}...")
    with open(ds['json_path'], 'r') as f:
        data = json.load(f)

    # Grab categories from the first file
    if ds_idx == 0:
        merged_data["categories"] = data.get("categories", [])

    img_id_mapping = {}

    # Process Images
    for img in data.get("images", []):
        old_img_id = img["id"]
        old_file_name = img["file_name"]

        # Prefix the file name with the dataset index to prevent collisions
        new_file_name = f"ds{ds_idx}_{old_file_name}"

        # Map IDs
        img_id_mapping[old_img_id] = global_img_id

        # Update image info with new ID and name
        img["id"] = global_img_id
        img["file_name"] = new_file_name
        merged_data["images"].append(img)

        # Copy the physical image file to the new master folder
        src_path = os.path.join(ds["img_dir"], old_file_name)
        dst_path = os.path.join(output_img_dir, new_file_name)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
        else:
            print(f"  ⚠️ Warning: Could not find image {src_path}")

        global_img_id += 1

    # Process Annotations
    for ann in data.get("annotations", []):
        if ann["image_id"] not in img_id_mapping:
            continue

        ann["id"] = global_ann_id
        ann["image_id"] = img_id_mapping[ann["image_id"]]
        merged_data["annotations"].append(ann)

        global_ann_id += 1

# Save the master JSON
with open(output_json, 'w') as f:
    json.dump(merged_data, f, indent=4)

print(f"\n✅ Merge complete! Saved {len(merged_data['images'])} images and {len(merged_data['annotations'])} annotations.")
print(f"Your new dataset is ready in the '{master_folder}' directory!")

Processing images400.json...
Processing instances_default.json...
Processing annotations.json...
Processing annotations_fixed.json...

✅ Merge complete! Saved 988 images and 3763 annotations.
Your new dataset is ready in the 'my_merged_coco_dataset' directory!
